## Step 1: Install & Import Libraries

In [2]:
!pip install langchain
!pip install langchain-groq
!pip install langchain-community beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
!pip install openai openrouter python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 60.8 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.5
    Uninstalling pydantic_core-2.46.5:
      Successfully uninstalled pydantic_core-2.46.5
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.5
    Uninstalling pydantic-2.13.5:
      Successfully uninstalled pydantic-2.13.5


In [4]:
import os
import pandas as pd
from tqdm import tqdm
import json
import re

## Step 2: Set the API Keys

In [5]:
os.environ["OPENAI_API_KEY"] = "Put your key here"
os.environ["GROQ_API_KEY"] = "Put your key here"
os.environ["OPENROUTER_API_KEY"] = "Put your key here"

##  Step 3: Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [7]:
MentalQA = pd.read_excel('Entities&Relations_MentalQA_500_data.xlsx')

In [8]:
MentalQA.head()

,question,answer,final_QT,final_AS
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,سد تى نعم ان هناك العديد من القلق النفسى المرت...,"['A', 'D']","['1', '2']"
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,يجب العرض علي طبيب اختصاصي امراض نفسية سوف يقو...,"['B', 'E']",['2']
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,الوسواس القهري هو من الامراض الشاءعة ويكون في ...,"['A', 'E']","['1', '2']"
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,العلاج النفسي المعرفي السلوكي يعطي نتائج جيدة ...,"['B', 'D']",['1']
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,ثابري مع طبيبك واخبريه بمخاوفك والتزمي بالجوان...,"['A', 'B', 'D']","['2', '3']"


In [9]:
MentalQA.columns

Index(['question', 'answer', 'final_QT', 'final_AS'], dtype='object')

In [10]:
df = MentalQA.copy()

In [11]:
df_q = df[["question"]]

## Step 4: Define a Shared Arabic NER Prompt

### 4-shot

In [ ]:
def build_prompt(question):
    return f"""

أنت خبير في المعالجة اللغوية للنصوص الطبية باللغة العربية، ومهمتك تنفيذ عملية الوسم الدلالي
(Named Entity Recognition - NER)
.لاستخراج كيانات طبية محددة من استفسارات يطرحها المرضى في سياق الصحة النفسية

تتمثل مهمتك في تحديد واستخراج الكيانات التالية فقط، كما وردت نصًا في الاستفسار:

- Symptom: الأعراض النفسية أو السلوكية الظاهرة
- Diagnosis: الحالات أو الاضطرابات النفسية المشخّصة أو المشار إليها
- Treatment: طرق أو أنواع العلاج غير الدوائية (مثل: العلاج السلوكي المعرفي، الاستشارة النفسية)
- Drug: أسماء الأدوية أو العقاقير
- Dosage: معلومات عن الجرعة (مثلًا: مرتين في اليوم، 50 ملغ)
- Duration: مدة الاستخدام أو استمرار الحالة (مثلًا: منذ أسبوع، لمدة 3 أشهر)
- Strength: قوة أو تركيز الدواء (مثلًا: 500، 1.0)
- Unit: وحدة قياس التركيز (مثلًا: ملغ، g، ml)
- Risk: مؤشرات على وجود خطر على السلامة النفسية أو الجسدية (مثل الانتحار أو إيذاء النفس)

تعليمات مهمة:
- استخرج فقط الكيانات المذكورة بشكل صريح في نص الاستفسار.
- استخدم الصياغة الحرفية كما وردت، سواء كانت بالعربية أو الإنجليزية.
- اكتب النتيجة بصيغة قائمة واحدة تحتوي على جميع الكيانات، بهذا التنسيق:
[{{"Symptom": "...", "Diagnosis": "...", "Treatment": "...", "Drug": "...", "Dosage": "...", "Duration": "...", "Strength": "...", "Unit": "...", "Risk": "..."}}]
- إذا لم يُذكر كيان معين، اجعل قيمته: ""
- لا تضف أي شرح أو تعليق إضافي فقط أعد الكيانات بالتنسيق المطلوب.

أمثلة:

مثال ١:
السؤال: كيف انتقل من دواء السيروكسات الذي اتناول منه جرعة70ملغم يوميا منذ عام الى السيبرالكس لاتفادى الاعراض الانحسابية علما ان الطبيب المعالج طلب مني ان استبدل السيروكسات بالسيبرالكس
الإخراج:
[{{"Symptom": "", "Diagnosis": "", "Treatment": "استبدل", "Drug": "السيروكسات، السيبرالكس", "Dosage": "70ملغم يوميا", "Duration": "منذ عام", "Strength": "70", "Unit": "ملغم", "Risk": ""}}]

مثال ٢:
السؤال: بدأت حديثا بتناول دواء سيروكسات 20 لمعالجة الرهاب الاجتماعي بمقدار نصف حبة يوميا لكن ظهرت لدي مشكلة كثرة التثاؤب ,هل ستبقى هذه المشكلة طيلة فترة العلاج او فقط في البداية ؟ كيف اتخلص منها ؟
الإخراج:
[{{"Symptom": "كثرة التثاؤب", "Diagnosis": "الرهاب الاجتماعي", "Treatment": "معالجة", "Drug": "سيروكسات", "Dosage": "نصف حبة يوميا", "Duration": "", "Strength": "20", "Unit": "", "Risk": ""}}]

مثال ٣:
السؤال: من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و ما اعرف كيف اتخلص من ذي العادة, و بدت تجيني افكار بإنهاء حياتي و حاولت انتحر باكثر من مرة و أكثر من طريقة
الإخراج:
[{{"Symptom": "أذي نفسي", "Diagnosis": "", "Treatment": "", "Drug": "", "Dosage": "", "Duration": "من سنه", "Strength": "", "Unit": "", "Risk": "أذي نفسي، حاولت انتحر، افكار بإنهاء حياتي"}}]

مثال ٤:
السؤال: هل الانتحار عن طريق حبوب المنوم يجعلني اشعر بالراحة ولا يكون مؤلم؟
الإخراج:
[{{"Symptom": "", "Diagnosis": "", "Treatment": "", "Drug": "حبوب المنوم", "Dosage": "", "Duration": "", "Strength": "", "Unit": "", "Risk": "الانتحار، الانتحار عن طريق حبوب المنوم"}}]

الآن استخرج الكيانات من المثال التالي:

السؤال: {question}
"""


### Zero-shot

In [12]:
def build_prompt(question):
    return f"""

أنت خبير في المعالجة اللغوية للنصوص الطبية باللغة العربية، ومهمتك تنفيذ عملية الوسم الدلالي
(Named Entity Recognition - NER)
.لاستخراج كيانات طبية محددة من استفسارات يطرحها المرضى في سياق الصحة النفسية

تتمثل مهمتك في تحديد واستخراج الكيانات التالية فقط، كما وردت نصًا في الاستفسار:

- Symptom: الأعراض النفسية أو السلوكية الظاهرة
- Diagnosis: الحالات أو الاضطرابات النفسية المشخّصة أو المشار إليها
- Treatment: طرق أو أنواع العلاج غير الدوائية (مثل: العلاج السلوكي المعرفي، الاستشارة النفسية)
- Drug: أسماء الأدوية أو العقاقير
- Dosage: معلومات عن الجرعة (مثلًا: مرتين في اليوم، 50 ملغ)
- Duration: مدة الاستخدام أو استمرار الحالة (مثلًا: منذ أسبوع، لمدة 3 أشهر)
- Strength: قوة أو تركيز الدواء (مثلًا: 500، 1.0)
- Unit: وحدة قياس التركيز (مثلًا: ملغ، g، ml)
- Risk: مؤشرات على وجود خطر على السلامة النفسية أو الجسدية (مثل الانتحار أو إيذاء النفس)

تعليمات مهمة:
- استخرج فقط الكيانات المذكورة بشكل صريح في نص الاستفسار.
- استخدم الصياغة الحرفية كما وردت، سواء كانت بالعربية أو الإنجليزية.
- اكتب النتيجة بصيغة قائمة واحدة تحتوي على جميع الكيانات، بهذا التنسيق:
[{{"Symptom": "...", "Diagnosis": "...", "Treatment": "...", "Drug": "...", "Dosage": "...", "Duration": "...", "Strength": "...", "Unit": "...", "Risk": "..."}}]
- إذا لم يُذكر كيان معين، اجعل قيمته: ""
- لا تضف أي شرح أو تعليق إضافي فقط أعد الكيانات بالتنسيق المطلوب.



**السؤال**:

{question}


"""

## Step 5: Define Helper Functions for Each Model

### GPT-4o (via OpenAI)

In [13]:
import openai

def call_gpt4o(prompt):
    client = openai.OpenAI(
        api_key=os.environ["OPENAI_API_KEY"])
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]

    )
    return response.choices[0].message.content

### LLLaMA (via Groq)

In [ ]:
from langchain_groq import ChatGroq

llama_chat = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.3-70b-versatile"     # "meta-llama/llama-4-scout-17b-16e-instruct" <-------- Have been deprecation by Qroq
)

In [ ]:
def call_llama(prompt):
    try:
        response = llama_chat.invoke([
            {"role": "user", "content": prompt}
        ])
        return response.content
    except Exception as e:
        return f"LLaMA Error: {e}"

### LLLaMA (via OpenRouter)

In [14]:
from openai import OpenAI

llama_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [15]:
def call_llama(prompt):
    try:
        response = llama_client.chat.completions.create(
            model="meta-llama/llama-4-scout",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"LLaMA Error: {e}"

## Step 6: Run

### Run

In [16]:
!pip install tqdm

In [17]:
results_dict = {}

for i, row in df_q.iterrows():
    key = row['question']
    results_dict[key] = {
        "question": row["question"]
    }

In [18]:
from tqdm.auto import tqdm
import json
import time
import sys
import os

batch_size = 100
sleep_time = 1
total_rows = len(df_q)

### GPT-4o

In [19]:
gpt_results_path = "gpt4o_outputs.json"

In [20]:
for batch_start in range(0, total_rows, batch_size):
    batch_end = min(batch_start + batch_size, total_rows)
    print(f"\nProcessing batch {batch_start} → {batch_end}")

    batch_df = df_q.iloc[batch_start:batch_end].copy()

    progress_bar = tqdm(
        batch_df.iterrows(),
        total=len(batch_df),
        desc=f"Batch {batch_start // batch_size + 1}",
        position=0,
        dynamic_ncols=True,
        file=sys.stdout,
        mininterval=0.5,
        leave=True
    )

    for i, row in progress_bar:
        key = row["question"]

        # Skip if already annotated by GPT-4o
        if "GPT-4o Output" in results_dict.get(key, {}):
            continue

        try:
            prompt = build_prompt(row["question"])
            gpt_output = call_gpt4o(prompt)
            time.sleep(sleep_time)
        except Exception as e:
            gpt_output = f"GPT-4o Error: {e}"
            with open(gpt_results_path, "w", encoding="utf-8") as f:
                json.dump(results_dict, f, ensure_ascii=False, indent=2)
            print(f"\nError at row {i + batch_start}: {e}")
            break

        # Create entry if it doesn't exist
        if key not in results_dict:
            results_dict[key] = {"question": row["question"]}

        results_dict[key]["GPT-4o Output"] = gpt_output

        # Save after each successful row
        with open(gpt_results_path, "w", encoding="utf-8") as f:
            json.dump(results_dict, f, ensure_ascii=False, indent=2)



Processing batch 0 → 100


Batch 1:   0%|          | 0/100 [00:00<?, ?it/s]


Processing batch 100 → 200


Batch 2:   0%|          | 0/100 [00:00<?, ?it/s]


Processing batch 200 → 300


Batch 3:   0%|          | 0/100 [00:00<?, ?it/s]


Processing batch 300 → 400


Batch 4:   0%|          | 0/100 [00:00<?, ?it/s]


Processing batch 400 → 499


Batch 5:   0%|          | 0/99 [00:00<?, ?it/s]

#### GPT-4o Results

In [21]:
#gpt4o_df = pd.DataFrame(results_dict.values())
gpt4o_df_zeroshot = pd.DataFrame(results_dict.values())

In [22]:
gpt4o_df_zeroshot.tail()

,question,GPT-4o Output
494,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...,"[{""Symptom"": ""قلق"", ""Diagnosis"": ""وسواس قهري"",..."
495,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...,"[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm..."
496,لدي خمس سنوات علاج من الذهانية ولم اشفى تماما ...,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre..."
497,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...,"[{""Symptom"": ""فقدان التركيز"", ""Diagnosis"": """",..."
498,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...,"[{""Symptom"": ""ماعندي شهيه"", ""Diagnosis"": ""الاك..."


In [24]:
from google.colab import files
gpt4o_df_zeroshot.to_excel('gpt4o_df_zeroshot.xlsx')
files.download('gpt4o_df_zeroshot.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### LLaMA

In [25]:
# Parameters
batch_size = 100
sleep_time = 1
results_path = "llama_outputs.json"
total_rows = len(df_q)

In [26]:
# Loop through dataset in batches
for batch_start in range(0, total_rows, batch_size):
    batch_end = min(batch_start + batch_size, total_rows)
    print(f"\nProcessing batch {batch_start} → {batch_end}")

    batch_df = df_q.iloc[batch_start:batch_end].copy()

    progress_bar = tqdm(
        batch_df.iterrows(),
        total=len(batch_df),
        desc=f"Batch {batch_start // batch_size + 1}",
        position=0,
        dynamic_ncols=True,
        file=sys.stdout,
        mininterval=0.5,
        leave=True
    )

    for i, row in progress_bar:
        key = row["question"]

        if "LLaMA Output" in results_dict.get(key, {}):
            continue

        try:
            prompt = build_prompt(row["question"])
            llama_output = call_llama(prompt)
            time.sleep(sleep_time)
        except Exception as e:
            llama_output = f"LLaMA Error: {e}"
            with open(results_path, "w", encoding="utf-8") as f:
                json.dump(results_dict, f, ensure_ascii=False, indent=2)
            print(f"\nError at row {i + batch_start}: {e}")
            break

        if key not in results_dict:
            results_dict[key] = {"question": row["question"]}

        results_dict[key]["LLaMA Output"] = llama_output

        # Save after each successful row
        with open(results_path, "w", encoding="utf-8") as f:
            json.dump(results_dict, f, ensure_ascii=False, indent=2)


Processing batch 0 → 100


Batch 1:   0%|          | 0/100 [00:00<?, ?it/s]


Processing batch 100 → 200


Batch 2:   0%|          | 0/100 [00:00<?, ?it/s]


Processing batch 200 → 300


Batch 3:   0%|          | 0/100 [00:00<?, ?it/s]


Processing batch 300 → 400


Batch 4:   0%|          | 0/100 [00:00<?, ?it/s]


Processing batch 400 → 499


Batch 5:   0%|          | 0/99 [00:00<?, ?it/s]

#### LLaMA Results

In [27]:
#llama_df = pd.DataFrame(results_dict.values())
llama_df_zeroshot = pd.DataFrame(results_dict.values())

In [28]:
llama_df_zeroshot.tail()

,question,GPT-4o Output,LLaMA Output
494,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...,"[{""Symptom"": ""قلق"", ""Diagnosis"": ""وسواس قهري"",...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ..."
495,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...,"[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","[{""Symptom"": ""الخوف، تنبؤات من كلام الأهل""، ""D..."
496,لدي خمس سنوات علاج من الذهانية ولم اشفى تماما ...,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre..."
497,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...,"[{""Symptom"": ""فقدان التركيز"", ""Diagnosis"": """",...","[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ..."
498,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...,"[{""Symptom"": ""ماعندي شهيه"", ""Diagnosis"": ""الاك...","[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، فقد..."


In [29]:
from google.colab import files
llama_df_zeroshot.to_excel('llama_df_zeroshot.xlsx')
files.download('llama_df_zeroshot.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>